# Info sobre outputs
Esse arquivo r

# Importando variáveis

In [ ]:
from 02_distancia_vias_e_ind.ipynb import stations

## 7.1 Montando tabelas de referência

In [13]:
# Lendo os arquivos para cada tabela
# https://www.gov.br/mma/pt-br/assuntos/meio-ambiente-urbano-recursos-hidricos-qualidade-ambiental/qualidade-do-ar/guia-tecnico-para-o-monitoramento-e-avaliacao-da-qualidade-do-ar.pdf

ref_table_co = pd.read_csv(inputs_path + '/ref_table_so2eco.csv')
ref_table_no2 = pd.read_csv(inputs_path + '/ref_table_no2.csv')
ref_table_o3 = pd.read_csv(inputs_path + '/ref_table_o3.csv')
ref_table_pm = pd.read_csv(inputs_path + '/ref_table_pm.csv')
ref_table_so2 = pd.read_csv(inputs_path + '/ref_table_so2eco.csv')

# Nota da EPA acerca da ocorrência de valores intermediários de ADT
'''
Distance from the edge of the nearest traffic lane. The distance for 
intermediate traffic counts should be interpolated from the table values based
on the actual traffic count.

Distância da borda da via mais próxima. A distância para contagens de veículos (ADT)
intermediárias deve ser interpolada a partir dos valores das tabelas baseados na
contagem de veículos observada.

# https://www.ecfr.gov/current/title-40/chapter-I/subchapter-C/part-58/appendix-Appendix%20E%20to%20Part%2058
'''

# Dicionário de faixas de ADT (real = valor * 1000)
pollutants_adt_dict = {'co':[1, 10, 20, 30, 40, 50, 60, np.inf],
                       'so2':[1, 10, 20, 30, 40, 50, 60, np.inf],
                       'no2':[1, 10, 15, 20, 40, 70, 110, np.inf],
                       'pm': [1, 15, 20, 30, 40, 50, 60, 70, 80, np.inf],
                       'o3':[10, 15, 20, 40, 70, 110, np.inf]}

# Definindo coluna de adt como índice
interpolated_co = ref_table_co.set_index('avg_adt').squeeze()
interpolated_no2 = ref_table_no2.set_index('avg_adt').squeeze()
interpolated_o3 = ref_table_o3.set_index('avg_adt').squeeze()
interpolated_pm = ref_table_pm.set_index('avg_adt').squeeze()
interpolated_so2 = ref_table_so2.set_index('avg_adt').squeeze()


## 7.2 Interpolação dos limites de distância para cada classe de representatividade espacial

In [14]:
# Montando dicionário de dataframes para interpolação
interpolated_dict = {
    'co': interpolated_co,
    'so2': interpolated_so2,
    'no2': interpolated_no2,
    'pm': interpolated_pm,
    'o3': interpolated_o3
}

# Redefinindo dicionário de subsets de poluentes
pollutant_subsets = {
    'co': subset_co,
    'so2': subset_so2,
    'no2': subset_no2,
    'pm': subset_pm,
    'o3': subset_o3
}

## Interpolando os limites das classes de representatividade ------------------------
# Iterando sobre os poluentes e os subsets de poluentes
for poll, subset in pollutant_subsets.items():
    
    # Iterando sobre os valores de adt da tabela de referencia de cada poluente
    for idx, adt_band in enumerate(pollutants_adt_dict[poll][:-1]):
        col_name = f'average_daily_vehicle_count_{adt_band}k'
        
        if col_name not in subset.columns:
            print(f"[WARNING] '{col_name}' nonexistant in subset_{poll}")
            continue
        
        # Iterando sobre os valores de adt para cada via do subset do poluente
        for adt_value in subset[col_name]:
            interpolated_dict[poll].loc[adt_value] = np.nan
                
    # Organizar pelo índice de modo ascendente
    interpolated_dict[poll].sort_index(inplace=True)
                
    # Interpolando os valores NaN #FIXME
    interpolated_dict[poll].interpolate(method='index',
                                        inplace=True)
                
    # Resetando index
    interpolated_dict[poll].reset_index(inplace=True)

/tmp/ipykernel_489887/3179769829.py:39: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  interpolated_dict[poll].interpolate(method='index',
/tmp/ipykernel_489887/3179769829.py:39: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  interpolated_dict[poll].interpolate(method='index',
/tmp/ipykernel_489887/3179769829.py:39: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  interpolated_dict[poll].interpolate(method='index',
/tmp/ipykernel_489887/3179769829.py:39: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead